# AffectLab IEMOCAP calibrated late fusion

This analysis pairs the frozen five-fold context-text and audio predictions by utterance ID. It averages their validation-calibrated posteriors with a predeclared 0.5/0.5 weight, then compares fusion against context text using a paired 10,000-sample dialogue-cluster bootstrap. No GPU is required.

In [ ]:
import subprocess
from google.colab import auth

PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'affectlab-research-raluca-biras'
TEXT_EXPERIMENT = 'iemocap_benchmark4_context3_deberta_v3_small'
AUDIO_EXPERIMENT = 'iemocap_benchmark4_audio_wav2vec2_base'
FUSION_EXPERIMENT = 'iemocap_benchmark4_context3_audio_equal_fusion'
auth.authenticate_user()
subprocess.run(['gcloud', 'config', 'set', 'project', PROJECT_ID], check=True)

In [ ]:
import base64
import json
import os
from pathlib import Path
from google.colab import userdata

REPO_URL = 'https://github.com/ralucabiras/emotion-aware-role-play-model.git'
REPO_DIR = Path('/content/emotion-aware-role-play-model')
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets and enable notebook access.')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
auth_option = f'http.extraHeader=Authorization: Basic {basic_auth}'
if not REPO_DIR.exists():
    subprocess.run(['git', '-c', auth_option, 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), '-c', auth_option, 'pull', '--ff-only'], check=True)
del github_token, basic_auth, auth_option
os.chdir(REPO_DIR)

In [ ]:
INPUT_ROOT = Path('/content/iemocap-fusion-input')
TEXT_DIR = INPUT_ROOT / TEXT_EXPERIMENT
AUDIO_DIR = INPUT_ROOT / AUDIO_EXPERIMENT
OUTPUT_DIR = Path('/content/iemocap-fusion-output') / FUSION_EXPERIMENT
for fold in range(1, 6):
    for experiment, local_dir, family in ((TEXT_EXPERIMENT, TEXT_DIR, 'iemocap-text'), (AUDIO_EXPERIMENT, AUDIO_DIR, 'iemocap-audio')):
        fold_dir = local_dir / f'fold-{fold}'
        fold_dir.mkdir(parents=True, exist_ok=True)
        for filename in ('metrics.json', 'test_predictions.jsonl'):
            source = f'gs://{BUCKET}/runs/{family}/{experiment}/fold-{fold}/{filename}'
            subprocess.run(['gcloud', 'storage', 'cp', source, str(fold_dir / filename)], check=True)
print('Downloaded paired predictions only; model checkpoints were not copied.')

In [ ]:
subprocess.run([
    'python', '-m', 'ml.evaluation.fuse_iemocap_modalities',
    '--text-dir', str(TEXT_DIR), '--audio-dir', str(AUDIO_DIR),
    '--output-dir', str(OUTPUT_DIR), '--text-weight', '0.5',
    '--iterations', '10000', '--seed', '20260808',
], check=True)
result = json.loads((OUTPUT_DIR / 'summary.json').read_text())
display({
    'method': result['method'],
    'weights': {'text': result['text_weight'], 'audio': result['audio_weight']},
    'pooled': result['pooled'],
    'macro_f1_comparison': result['paired_dialogue_bootstrap']['observed']['macro_f1'],
})

In [ ]:
destination = f'gs://{BUCKET}/runs/iemocap-fusion/{FUSION_EXPERIMENT}'
subprocess.run(['gcloud', 'storage', 'rsync', '--recursive', str(OUTPUT_DIR), destination], check=True)
print('Private fusion artifacts uploaded to:', destination)

## Interpretation guardrails

- Treat the fixed 0.5/0.5 result as the confirmatory fusion comparison.
- Do not optimize modality weights on these held-out predictions.
- Use the dialogue-bootstrap interval, not only the point estimate, when claiming improvement.
- Keep all row-level IEMOCAP predictions private.